In [13]:
import gymnasium as gym
import numpy as np
import cv2
import tensorflow as tf
from keras.layers import Dense , Input
from keras import Model

env = gym.make("CartPole-v1")

In [14]:

# ==========================================
# Q-Network Definition
# ==========================================
net_input = Input(shape=(4,))
x = Dense(64 , activation='relu')(net_input)
x = Dense(32 , activation='relu')(x)
net_output = Dense(2 , activation='linear')(x)
q_net = Model(inputs=net_input , outputs=net_output)


In [15]:

# ==========================================
# Parameters
# ==========================================
GAMMA = 0.99
ALPHA = 0.001
EPSILON = 1.0
EPSILON_DECAY = 1.001
NUM_EPISODES = 500


In [16]:

# ==========================================
# Policy Function
# ==========================================
def policy(state, explore=0.0):
    # Epsilon-Greedy policy
    if tf.random.uniform(shape=(), maxval=1.0) <= explore:
        return np.random.randint(0, 2)
    
    # State already (1, 4) form mein aayega
    q_values = q_net(state)
    return int(tf.argmax(q_values[0]).numpy())


In [17]:

# ==========================================
# Custom Training Loop (SARSA with TF Gradients)
# ==========================================
for episode in range(NUM_EPISODES):
    total_reward = 0
    length_episode = 0
    
    # 1. Reset env aur Shape ko (4,) se (1, 4) banayein
    state, info = env.reset()
    state = tf.expand_dims(tf.convert_to_tensor(state, dtype=tf.float32), 0)
    
    action = policy(state, explore=EPSILON)
    
    done = False
    while not done:
        # 2. Gym step (Unpacking 5 values)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        # Shape change to (1, 4) for NN
        next_state = tf.expand_dims(tf.convert_to_tensor(next_state, dtype=tf.float32), 0)
        
        # Agla action choose karein
        next_action = policy(next_state, explore=EPSILON)
        
        # 3. Target Calculation (Fixed typos)
        if done:
            target = float(reward)
        else:
            target = reward + GAMMA * q_net(next_state)[0][next_action].numpy()
        
        # 4. Gradient Calculation (Sirf chune gaye action ki Q-value par)
        with tf.GradientTape() as tape:
            current_q_values = q_net(state)
            current_q = current_q_values[0][action] # Scalar Q-value for chosen action
            
        grads = tape.gradient(current_q, q_net.trainable_weights)
        
        # Delta = Target - Current
        delta = target - current_q.numpy()
        
        # 5. Weight Update Rule (Manual Gradient Ascent)
        for j in range(len(grads)):
            q_net.trainable_weights[j].assign_add(ALPHA * delta * grads[j])
            
        # Agle step ke liye variables update
        state = next_state
        action = next_action
        
        # 6. Correct reward assignment
        total_reward += reward
        length_episode += 1
    
        
    # Print episode ke baad hoga
    print(f"Episode: {episode + 1}, Length: {length_episode}, Total Reward: {total_reward}, Epsilon: {EPSILON:.3f}")
    
    EPSILON /= EPSILON_DECAY  # Decay epsilon after each episode

env.close()

Episode: 1, Length: 16, Total Reward: 16.0, Epsilon: 1.000
Episode: 2, Length: 43, Total Reward: 43.0, Epsilon: 0.999
Episode: 3, Length: 19, Total Reward: 19.0, Epsilon: 0.998
Episode: 4, Length: 24, Total Reward: 24.0, Epsilon: 0.997
Episode: 5, Length: 12, Total Reward: 12.0, Epsilon: 0.996
Episode: 6, Length: 14, Total Reward: 14.0, Epsilon: 0.995
Episode: 7, Length: 26, Total Reward: 26.0, Epsilon: 0.994
Episode: 8, Length: 16, Total Reward: 16.0, Epsilon: 0.993
Episode: 9, Length: 12, Total Reward: 12.0, Epsilon: 0.992
Episode: 10, Length: 27, Total Reward: 27.0, Epsilon: 0.991
Episode: 11, Length: 44, Total Reward: 44.0, Epsilon: 0.990
Episode: 12, Length: 28, Total Reward: 28.0, Epsilon: 0.989
Episode: 13, Length: 31, Total Reward: 31.0, Epsilon: 0.988
Episode: 14, Length: 25, Total Reward: 25.0, Epsilon: 0.987
Episode: 15, Length: 15, Total Reward: 15.0, Epsilon: 0.986
Episode: 16, Length: 15, Total Reward: 15.0, Epsilon: 0.985
Episode: 17, Length: 23, Total Reward: 23.0, Epsi

In [18]:
q_net.save("sarsa_cartpole_model.h5")
print("Model saved as 'sarsa_cartpole_model.h5'")

Model saved as 'sarsa_cartpole_model.h5'
